In [2]:
import json
import os
from collections import Counter
from pathlib import Path

cwd = Path(os.getcwd())
print("当前工作目录：", cwd)

# 当前在 notebooks/notebooks，向上回退2层，回到【第二周】根目录
root_dir = cwd.parent.parent

DATA_PATH = root_dir / "data" / "week02_sentences.txt"
OUTPUT_PATH = root_dir / "outputs" / "week02_stats.json"
OUTPUT_PATH.parent.mkdir(exist_ok=True)

print(f"数据文件完整路径: {DATA_PATH}")
print(f"文件是否存在: {DATA_PATH.exists()}")

当前工作目录： c:\Users\wenliy\Desktop\第二周\notebooks\notebooks
数据文件完整路径: c:\Users\wenliy\Desktop\第二周\data\week02_sentences.txt
文件是否存在: True


In [3]:
def load_lines(path: Path) -> list[str]:
    """
    读取文本文件，按行返回句子列表，去除每行首尾空白，跳过空行
    path：文件路径Path对象
    return：句子列表
    """
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            text = line.strip()
            if text: 
                lines.append(text)
    return lines


def normalize_text(text: str) -> str:
    """
    文本归一化清洗：去除空格，转小写（英文生效），简单预处理
    """
    text = text.strip()
    text = text.lower()
    return text


def count_tokens(lines: list[str]) -> Counter:
    """
    简易分词统计：中文按字符，英文按空格切分，返回Counter词频
    """
    all_tokens = []
    for line in lines:
        clean = normalize_text(line)
        tokens = list(clean)
        all_tokens.extend(tokens)
    return Counter(all_tokens)


# 测试load_lines
test_lines = load_lines(DATA_PATH)
print(f"load_lines测试，读取到句子数量：{len(test_lines)}")

# 测试normalize_text
test_text = "  HELLO Python！测试文本  "
print(f"normalize_text输入：{test_text}，输出：{normalize_text(test_text)}")

# 测试count_tokens
test_sample = ["我爱python", "python很好用"]
cnt = count_tokens(test_sample)
print("count_tokens测试，前5个高频：", cnt.most_common(5))

load_lines测试，读取到句子数量：30
normalize_text输入：  HELLO Python！测试文本  ，输出：hello python！测试文本
count_tokens测试，前5个高频： [('p', 2), ('y', 2), ('t', 2), ('h', 2), ('o', 2)]


In [4]:
#读取全部句子
sentences = load_lines(DATA_PATH)
num_sentences = len(sentences)

#计算每条句子字符长度
sent_len = [len(s) for s in sentences]
avg_length = sum(sent_len) / num_sentences

# 拿到最长5条句子
# 将句子和长度配对，按长度降序排序
sent_with_len = list(zip(sentences, sent_len))
sent_with_len.sort(key=lambda x:x[1], reverse=True)
longest_examples = [item[0] for item in sent_with_len[:5]]

#统计字符频次，取top20
token_counter = count_tokens(sentences)
top_tokens = dict(token_counter.most_common(20))

print(f"句子总数：{num_sentences}")
print(f"平均字符长度：{avg_length:.2f}")
print("最长5句：")
for s in longest_examples:
    print("-", s)
print("top20字符：", top_tokens)

句子总数：30
平均字符长度：16.77
最长5句：
- 这部电影的剧情节奏紧凑，全程没有多余桥段。
- 配乐搭配画面恰到好处，情绪渲染很到位。
- 整体完成度很高，是一部值得二刷的电影。
- 演员演技自然细腻，人物塑造非常立体。
- 剧情反转出乎意料，看完让人回味无穷。
top20字符： {'，': 30, '。': 30, '很': 13, '情': 9, '体': 8, '人': 8, '感': 7, '部': 7, '常': 6, '观': 6, '影': 6, '有': 6, '看': 6, '分': 6, '非': 5, '剧': 5, '节': 5, '得': 5, '画': 4, '面': 4}


In [5]:
result = {
    "num_sentences": num_sentences,
    "avg_length": round(avg_length, 2),
    "top_tokens": top_tokens,
    "longest_examples": longest_examples
}

with open(OUTPUT_PATH, "w", encoding="utf‑8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"统计结果已经保存至 {OUTPUT_PATH}")

#验证：重新读回来，确认文件正常可读
with open(OUTPUT_PATH, "r", encoding="utf‑8") as f:
    reload_data = json.load(f)
print("重新读取json成功，句子数量：", reload_data["num_sentences"])

统计结果已经保存至 c:\Users\wenliy\Desktop\第二周\outputs\week02_stats.json
重新读取json成功，句子数量： 30
